# R01 · Cookbook-Referenzlabor: Grundlagen: Installation, erste Datenbank, Datensätze, Zellen, Erweiterungen

**NHyDB-0.28.2-Notebook-Reihe · v1 · Referenzteil**

Diese Beispiele basieren auf dem offiziellen
[NHyDB Python Cookbook](https://github.com/nhydb-run4/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (Abschnitte 1–8), das als maßgebliche Referenz dient.
Code und erwartete Ausgaben sind unverändert aus dem Cookbook übernommen, die Begleittexte ins
Deutsche übersetzt; dieselben Beispiele führt auch die NHyDB-Testsuite aus.

* Jede Code-Zelle ist ein **eigenständiges Beispiel** – führe eine beliebige Zelle für sich aus oder nutze *Run All*.
* Jedes Beispiel legt seinen eigenen temporären Datenbankordner an.
* Unter jedem Beispiel findest du die im Cookbook abgedruckte Ausgabe zum Vergleich mit deiner.
* Die Hauptreihe (`../main/`) erklärt die Ideen hinter diesen APIs Schritt für Schritt.

**Inhalt** (18 ausführbare Beispiele)

* 1. Installation, Import und Version
* 2. Die Konzepte in fünf Minuten
* 3. Deine erste Datenbank
* 4. Datenbanken öffnen und Optionen
* 5. Commit, Sync, Close und Verhalten bei Abstürzen
* 6. Datensätze (CLASSIC-Layout)
* 7. Einzelne Zellen, Lanes, Null und Löschen
* 8. Erweiterungen: zusätzliche Felder pro Datensatz

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.28.2':
    print('⚠ These examples were written for NHyDB 0.28.2; results may differ.')

nhydb 0.28.2 · file format 0.28.2


## 1. Installation, Import und Version

```bash
python -m pip install nhydb            # prebuilt wheels: CPython 3.12–3.15 on Linux, macOS, Windows
python -m pip install numpy            # optional, for the NumPy batch API (section 16)
```

In [2]:
import nhydb

print(nhydb.ENGINE_VERSION)        # product version of the installed package
print(nhydb.FILE_FORMAT_VERSION)   # identifier of the on-disk format
assert nhydb.ENGINE_VERSION == "0.28.2"
assert nhydb.FILE_FORMAT_VERSION == "0.28.2"

0.28.2
0.28.2


*Erwartete Ausgabe (aus dem Cookbook):*

```text
0.28.2
0.28.2
```

`FILE_FORMAT_VERSION` bezeichnet das Dateiformat auf der Platte: das Engine-Release,
das es festgelegt hat. Die Datei beginnt mit der Signatur `NHYDB`, gefolgt von den
drei Versionsbytes 0, 28, 2. Datenbanken früherer Releases – NHyDB 0.28.0
(Dateikennung `NHYDB016`) und 0.27.1 oder älter – lassen sich nicht öffnen; sie lösen
`CorruptionError` aus, und nichts wird verändert (Abschnitt 21).

## 2. Die Konzepte in fünf Minuten

* **Zelle.** Jeder Wert liegt in einer Zelle `(R, u, v, lane)`: Realm `R`,
  Datensatz-Koordinate `u`, Feld/Position `v` und Lane `NL` (0) oder `NR` (1).
  Alle Koordinaten sind natürliche Zahlen ≥ 1 und dürfen beliebig groß sein.
* **N.** Jede Zelle hat eine eindeutige natürliche Zahl `N`, die aus der
  Adresse berechnet wird (Abschnitt 9). `N` wird nie gespeichert; es ist ein mathematischer
  Name der Zelle, über den du sie lesen kannst.
* **Realm.** Ein nummerierter Adressbereich (`create_realm`). Ein Realm fasst
  Shapes, Layouts, Erweiterungen und Indizes zusammen.
* **Shape.** Die BASE-Felder eines Datensatzes (`v = 1 … k`) mit Namen und Typen
  (`create_shape`).
* **Layout.** Wie logische Datensätze auf Zellen abgebildet werden. **CLASSIC** (Datensatz =
  `u`, Feld = `v`) ist der Normalfall; COLUMN, PACKED, CUSTOM und MATH
  werden in den Abschnitten 17 und 18 behandelt.
* **Dense und sparse.** CLASSIC-Datensätze (`put`, `put_record`,
  `put_dense_batch`, `put_numpy_batch`) werden *dense* (dicht) gespeichert; einzelne Zellen,
  Erweiterungen und Überschreibungen *sparse* (einzeln). Beim Lesen
  wird beides kombiniert; der neueste Schreibvorgang gewinnt.
* **Commit-Grenze.** Die Datei ist append-only. Daten werden mit
  `commit()` dauerhaft. Alles nach dem letzten Commit wird verworfen, wenn die
  Datenbank erneut geöffnet wird.
* **Dateien.** Eine Datenbank besteht aus `X.nhydb` (Daten),
  `X.nhydb.layout.enc` (verschlüsseltes Schema), `X.nhydb.layout.key` (Schlüssel – gut
  aufbewahren!) und `X.nhydb.lock` (Sperrdatei). Siehe Abschnitt 21.

## 3. Deine erste Datenbank

Lege eine Datenbank an, definiere einen Realm, ein Shape und ein CLASSIC-Layout, schreibe zwei
Datensätze, schließe die Datenbank, öffne sie wieder und lies.

In [3]:
import os
import tempfile

import nhydb

folder = tempfile.mkdtemp()
path = os.path.join(folder, "shop.nhydb")

db = nhydb.create(path)
db.create_realm(1, "shop")
db.create_shape(1, 1, "order", [
    {"v": 1, "name": "id", "type": "varuint"},
    {"v": 2, "name": "customer", "type": "utf8"},
    {"v": 3, "name": "amount", "type": "f64"},
])
db.create_classic_layout(1, 1, "orders", shape_id=1)   # layout 1 in realm 1 uses shape 1

db.put_record(1, 1, {"id": 1, "customer": "Alice", "amount": 19.90})
db.put_record(1, 2, {"id": 2, "customer": "Bob", "amount": 5.50})
db.commit()                                            # make it durable
db.close()

db = nhydb.open(path)
print(db.get_record(1, 1))
print(db.get_value(1, 2, 2))                           # realm 1, u=2, v=2 -> "customer"
db.close()

{'_realm': 1, '_u': 1, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 1, 'customer': 'Alice', 'amount': 19.9}
Bob


*Erwartete Ausgabe (aus dem Cookbook):*

```text
{'_realm': 1, '_u': 1, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 1, 'customer': 'Alice', 'amount': 19.9}
Bob
```

Felder, die mit `_` beginnen, beschreiben, wo der Datensatz liegt. Dasselbe Programm
mit einem Kontextmanager (`with` committet und schließt automatisch):

In [4]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "notes.nhydb")

with nhydb.create(path) as db:
    db.create_realm(1, "notes")
    db.create_shape(1, 1, "note", [{"v": 1, "name": "text", "type": "utf8"}])
    db.create_classic_layout(1, 1, "notes", shape_id=1)
    db.put_record(1, 1, {"text": "buy milk"})

with nhydb.open(path) as db:
    assert db.get_record(1, 1)["text"] == "buy milk"
    print(db.closed)
print(db.closed)

False
True


*Erwartete Ausgabe (aus dem Cookbook):*

```text
False
True
```

> **Hinweis:** `__exit__` ruft `close(commit=True)` auf, auch wenn im `with`-Block eine
> Ausnahme aufgetreten ist. Verwende ausdrücklich `close(commit=False)`, wenn du
> Änderungen nach einem Fehler verwerfen willst (Abschnitt 5).

## 4. Datenbanken öffnen und Optionen

`nhydb.open(path, **options)` ist dasselbe wie `nhydb.NHyDB(path, **options)`;
`nhydb.create(path, **options)` ist dasselbe wie `nhydb.NHyDB.create(...)`.
Alle Optionen sind Schlüsselwortargumente.

| Option | `create` | `open` | Bedeutung |
|---|:-:|:-:|---|
| `overwrite=False` | ✓ | | eine vorhandene Datenbank ersetzen |
| `durability="fsync"` | ✓ | ✓ | `"fsync"` oder `"full"` (Abschnitt 5) |
| `lock=True` | ✓ | ✓ | exklusive Sperrdatei `<db>.lock` (Abschnitt 22) |
| `lock_timeout_ms=0` | ✓ | ✓ | auf die Sperre warten, statt sofort zu scheitern |
| `layout_path`, `key_path` | ✓ | ✓ | andere Speicherorte für Schema- und Schlüsseldatei (Abschnitt 21) |
| `config` | ✓ | ✓ | Pfad einer `nhydb_config.toml` (Abschnitt 19) |
| `readonly=True` | | ✓ | öffnen, ohne die Datei zu verändern (keine Sperre, kein Commit; Schreib-APIs lösen `NHyDBError` aus) |

In [5]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "app.nhydb")
nhydb.create(path, durability="full").close()

try:
    nhydb.create(path)                       # already exists
except FileExistsError as e:
    print("FileExistsError")

db = nhydb.create(path, overwrite=True)      # start from scratch
print(str(db.path).endswith("app.nhydb"), db.durability)   # db.path is a pathlib.Path
db.close()

try:
    nhydb.open(os.path.join(os.path.dirname(path), "missing.nhydb"))
except FileNotFoundError:
    print("FileNotFoundError")

FileExistsError
True fsync
FileNotFoundError


*Erwartete Ausgabe (aus dem Cookbook):*

```text
FileExistsError
True fsync
FileNotFoundError
```

Nützliche Attribute einer geöffneten Datenbank:

In [6]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "info.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "main")
    db.commit()
    print(os.path.basename(db.path), os.path.basename(db.layout_path), os.path.basename(db.key_path))
    print(db.durability, db.max_db_size_bytes)
    print(db.committed_end > 0, db.recovered_tail_bytes, db.config_source)
    print(db.realms)

info.nhydb info.nhydb.layout.enc info.nhydb.layout.key
fsync 53687091200
True 0 None
{1: {'realm_id': 1, 'name': 'main', 'description': '', 'default_shape_id': None, 'base_reserve_to': 0}}


*Erwartete Ausgabe (aus dem Cookbook):*

```text
info.nhydb info.nhydb.layout.enc info.nhydb.layout.key
fsync 53687091200
True 0 None
{1: {'realm_id': 1, 'name': 'main', 'description': '', 'default_shape_id': None, 'base_reserve_to': 0}}
```

`db.durability` lässt sich auch bei geöffneter Datenbank ändern
(`db.durability = "full"`).

## 5. Commit, Sync, Close und Verhalten bei Abstürzen

* `commit()` schreibt alle ausstehenden Daten, synchronisiert die Datei und verschiebt die
  Commit-Grenze. Es gibt die neue Grenze zurück (einen Byte-Offset).
* Jede Schreibmethode hat standardmäßig `commit=False`; `commit=True` committet
  direkt nach dem Schreiben.
* `sync(hard=False)` schreibt Puffer in die Datei (`hard=True` synchronisiert
  die Datei zusätzlich), committet aber **nicht**.
* `close(commit=True)` committet und gibt Datei und Sperre frei.
  `close(commit=False)` schließt ohne Commit.
* Nach `close()` werden Datei-Handle und Sperre sofort freigegeben.
  Jeder Lese-, Schreib-, Abfrage- oder Index-Aufruf und jeder offene Iterator
  löst dann `NHyDBError("database is closed")` aus, auch für Daten, die vorher
  gelesen wurden; ein erneuter Aufruf von `close()` bewirkt nichts.
* Nicht committete Daten werden beim erneuten Öffnen der Datenbank verworfen;
  `recovered_tail_bytes` sagt dir, wie viele Bytes abgeschnitten wurden. Das geschieht
  nur, wenn die Commit-Grenze eindeutig ist: Ist einer der beiden Superblöcke
  beschädigt und folgen Daten hinter der Grenze, scheitert das Öffnen mit
  `CorruptionError` und verändert nichts (öffne mit `readonly=True` zum Lesen,
  löse die Lage mit `repair_file` – Abschnitt 20). Committete Daten werden nie
  stillschweigend abgeschnitten.

In [7]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "crash.nhydb")
db = nhydb.create(path)
db.create_realm(1, "r")
db.create_shape(1, 1, "s", [{"v": 1, "name": "n", "type": "varuint"}])
db.create_classic_layout(1, 1, "c", shape_id=1)
db.put_record(1, 1, {"n": 100}, commit=True)   # durable
db.put_record(1, 2, {"n": 200})                # not committed
db.close(commit=False)                         # like a crash before commit

db = nhydb.open(path)
print(db.recovered_tail_bytes > 0)             # the uncommitted tail was cut off
print(db.get_record(1, 1)["n"], db.get_record(1, 2))
db.close()
db.close()                                     # no effect

try:
    db.put_record(1, 3, {"n": 300})
except nhydb.NHyDBError as e:
    print(e)

True
100 None
database is closed


*Erwartete Ausgabe (aus dem Cookbook):*

```text
True
100 None
database is closed
```

**Dauerhaftigkeitsmodi.** `"fsync"` (Standard) nutzt den Datei-Sync des
Betriebssystems (unter macOS mit einer `F_BARRIERFSYNC`-Schreibbarriere vor dem
Superblock). `"full"` leert unter macOS zusätzlich den Cache des Laufwerks: Die
Daten werden mit `F_BARRIERFSYNC` vor dem Superblock eingeordnet, und der
Sync des Superblocks ist ein `F_FULLFSYNC`, das Daten und Superblock auf das
Medium schreibt, bevor `commit()` zurückkehrt; unter Linux und Windows führen beide
Modi denselben Aufruf aus.
`"full"` ist langsamer; unter macOS löst es einen Fehler aus, wenn das Dateisystem es
nicht unterstützt. Die Aliasse `"default"`/`"normal"` (= `"fsync"`) und
`"fullfsync"` (= `"full"`) werden akzeptiert. Scheitert ein Sync während eines Commits,
verweigert das Objekt weitere Commits; öffne die Datenbank neu.

**Rollback.** `rollback()` verwirft jeden Schreibvorgang und jede Schemaänderung
dieses Objekts seit seinem letzten Commit und gibt die Commit-Grenze zurück. Das
Objekt bleibt offen und nutzbar; Sitzungseinstellungen (registrierte Plugins,
`durability`) bleiben erhalten. Ohne ausstehende Änderungen bewirkt es nichts. Ein
Objekt mit `readonly=True` löst `NHyDBError` aus; mit `lock=False` wird zuerst
ein Commit eines anderen Schreibers erkannt und `DatabaseLockedError` ausgelöst (nichts
wird abgeschnitten). Schließen mit `commit=False` (oder das Ende des Prozesses) vor einem
Commit verwirft die Arbeit ebenfalls. Ein einzelner Aufruf, der mehrere Blöcke schreibt
(zum Beispiel `put_points` über viele `u`-Fenster), ist als Ganzes nicht atomar –
nutze `rollback()` nach einem Fehler oder verlass dich auf `put_math_batch`, das
sich selbst zurückrollt.

In [8]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "rollback.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "n", "type": "varuint"}])
    db.create_classic_layout(1, 1, "c", shape_id=1)
    db.put_record(1, 1, {"n": 100}, commit=True)
    boundary = db.committed_end

    db.put_record(1, 1, {"n": 111})                 # overwrite, not committed
    db.put_record(1, 2, {"n": 200})
    db.create_realm(2, "scratch")                   # schema change, not committed
    print(db.get_record(1, 1)["n"], sorted(db.realms))

    print(db.rollback() == boundary)                # back to the last commit
    print(db.get_record(1, 1)["n"], db.get_record(1, 2), sorted(db.realms))
    db.put_record(1, 3, {"n": 300}, commit=True)    # the handle stays usable

with nhydb.open(path, readonly=True) as db:
    print(db.get_record(1, 3)["n"])
    try:
        db.rollback()
    except nhydb.NHyDBError as e:
        print(e)

111 [1, 2]
True
100 None [1]
300
database is opened read-only (readonly=True); rollback is not possible


*Erwartete Ausgabe (aus dem Cookbook):*

```text
111 [1, 2]
True
100 None [1]
300
database is opened read-only (readonly=True); rollback is not possible
```

## 6. Datensätze (CLASSIC-Layout)

In einem CLASSIC-Layout ist ein Datensatz die Zellzeile `(R, u, v = 1 … k)`: Die
Datensatz-ID ist `u`, die Felder sind die Shape-Felder. Es gibt mehrere Wege,
Datensätze zu schreiben:

| Methode | Schreibt | Adressiert über |
|---|---|---|
| `put_record(layout_id, object_id, values_dict)` | einen Datensatz | Layout + Objekt-ID |
| `put(realm_id, u, values_list)` | einen Datensatz | Realm + `u` |
| `put_dense_batch(realm_id, u_start, rows)` | aufeinanderfolgende Datensätze `u_start, u_start+1, …` | Realm + `u` |
| `put_object` / `put_objects_batch` | einen / viele Datensätze **beliebiger** Layouts (Abschnitt 17) | Layout + Objekt-ID |

In [9]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "orders.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "shop")
    db.create_shape(1, 1, "order", [
        {"v": 1, "name": "id", "type": "varuint"},
        {"v": 2, "name": "customer", "type": "utf8"},
        {"v": 3, "name": "amount", "type": "f64"},
    ])
    db.create_classic_layout(1, 1, "orders", shape_id=1)

    db.put_record(1, 1, {"id": 1, "customer": "Alice", "amount": 19.9})      # dict by field name
    db.put(1, 2, [2, "Bob", 5.5], shape_id=1)                                  # list in field order
    db.put_dense_batch(1, 3, [[3, "Carol", 7.25], [4, "Dave", 120.0]], shape_id=1)  # u = 3, 4
    db.commit()

    print(db.get_record(1, 4))       # via layout 1, object 4
    print(db.get(1, 2))              # via realm 1, u = 2
    print(db.get_object(1, 3)["customer"])

{'_realm': 1, '_u': 4, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 4, 'customer': 'Dave', 'amount': 120.0}
{'_realm': 1, '_u': 2, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 2, 'customer': 'Bob', 'amount': 5.5}
Carol


*Erwartete Ausgabe (aus dem Cookbook):*

```text
{'_realm': 1, '_u': 4, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 4, 'customer': 'Dave', 'amount': 120.0}
{'_realm': 1, '_u': 2, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 2, 'customer': 'Bob', 'amount': 5.5}
Carol
```

`shape_id` kann entfallen, wenn der Realm eine Standard-Shape hat:
`create_classic_layout` setzt die Shape des Layouts als Standard, wenn der Realm
noch keine hat, und `set_realm_default_shape(realm_id, shape_id)` ändert sie.

**Überschreiben und löschen.** Schreibst du dasselbe `u` erneut, wird der Datensatz
ersetzt (der neueste Schreibvorgang gewinnt). `delete_record(layout_id, object_id)` löscht einen
Datensatz vollständig: jede lebende Zelle davon – die mit `put` geschriebenen Felder,
später mit `put_cell`, `put_cells_numpy` oder `batch()` geänderte Felder und
Erweiterungen. Es gibt die Zahl der gelöschten lebenden Zellen zurück (0, wenn es
nichts zu löschen gab). `delete_area(realm_id, u_from, u_to, v_from, v_to)`
löscht jede lebende Zelle eines Rechtecks und gibt ihre Zahl zurück; mit einem
`v`-Bereich, der alle Felder abdeckt, löscht es ganze Datensätze (Abschnitt 7).
Gelöschte Daten bleiben in der Append-only-Datei, bis `compact_file()` sie
physisch entfernt (Abschnitt 20).

In [10]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "people.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "people")
    db.create_shape(1, 1, "person", [
        {"v": 1, "name": "name", "type": "utf8"},
        {"v": 2, "name": "age", "type": "u8"},
    ])
    db.create_classic_layout(1, 1, "people", shape_id=1)
    db.put_dense_batch(1, 1, [["Ann", 31], ["Ben", 45], ["Cid", 27]], commit=True)

    db.put_record(1, 2, {"name": "Ben", "age": 46}, commit=True)   # birthday: replace u=2
    print(db.get_record(1, 2)["age"])

    print(db.delete_record(1, 1, commit=True))                      # live cells deleted: name, age
    print(db.get_record(1, 1))
    print(db.delete_area(1, 3, 3, 1, 2, commit=True), db.get(1, 3))  # u=3, v1..v2

46
2
None
2 None


*Erwartete Ausgabe (aus dem Cookbook):*

```text
46
2
None
2 None
```

**`put` ersetzt den Datensatz, `put_cell` ändert eine Zelle.** `put` (und
`put_dense_batch`, `put_numpy_batch`) schreibt den BASE-Datensatz der verwendeten Shape
als Ganzes; ein späteres `put` mit einer anderen Shape ersetzt ihn, sodass Felder, die
der neuen Shape fehlen, nicht mehr zum Datensatz gehören. `put_cell` ändert genau
eine Zelle und lässt die anderen stehen. Gelesen wird der Zustand, auf welchem Weg die Zelle
auch zuletzt geschrieben wurde – ein `put` nach einem `put_cell` liest das `put`. Zellen außerhalb der
Shape (Erweiterungen, reservierte BASE-Positionen, nur mit `put_cell` geschriebene
Zellen) folgen ihren eigenen Schreibvorgängen; ein `put` lässt sie in Ruhe.

In [11]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "replace.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "long", [{"v": 1, "name": "x", "type": "varint"}, {"v": 2, "name": "y", "type": "varint"}])
    db.create_shape(2, 1, "short", [{"v": 1, "name": "x", "type": "varint"}])
    db.create_classic_layout(1, 1, "c", shape_id=1)

    db.put(1, 1, {"x": 10, "y": 20}, shape_id=1)
    db.put(1, 1, {"x": 11}, shape_id=2)              # another shape: replaces the record
    print(db.get_value(1, 1, 1), db.get_cell(1, 1, 2, dyn_address=False)["present"])

    db.put(1, 2, {"x": 10, "y": 20}, shape_id=1)
    db.put_cell(1, 2, 1, 11)                         # one cell: y stays
    print(db.get_value(1, 2, 1), db.get_value(1, 2, 2))

    db.put_cell(1, 2, 1, 12)
    db.put(1, 2, {"x": 13, "y": 20}, shape_id=1)     # the newest write wins
    print(db.get_value(1, 2, 1))

11 False
11 20
13


*Erwartete Ausgabe (aus dem Cookbook):*

```text
11 False
11 20
13
```

**Enums.** Ein Feld kann Bezeichnungen auf ganzzahlige Codes abbilden. Du schreibst und liest
Bezeichnungen; die Datei speichert die Codes.

In [12]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "tickets.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "tickets")
    db.create_shape(1, 1, "ticket", [
        {"v": 1, "name": "title", "type": "utf8"},
        {"v": 2, "name": "state", "type": "u8", "enum": {"open": 0, "in_progress": 1, "done": 2}},
    ])
    db.create_classic_layout(1, 1, "tickets", shape_id=1)
    db.put_record(1, 1, {"title": "Fix login", "state": "in_progress"}, commit=True)
    print(db.get_record(1, 1)["state"])
    try:
        db.put_record(1, 2, {"title": "x", "state": "unknown"})
    except ValueError as e:
        print("ValueError:", e)

in_progress
ValueError: state: unknown enum value 'unknown'


*Erwartete Ausgabe (aus dem Cookbook):*

```text
in_progress
ValueError: state: unknown enum value 'unknown'
```

**Standard-Shape, Rohansichten und Dateipositionen.** Ein Realm kann mehrere
Shapes haben; `set_realm_default_shape` wählt dasjenige, das verwendet wird, wenn `shape_id`
fehlt. `get_internal` zeigt die Rohwerte nach `v`; `committed_end`,
`last_valid_end` (Ende der geschriebenen Daten, einschließlich nicht committeter Blöcke)
und `next_block_id` beschreiben den Zustand des Containers.

In [13]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "internal.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "small", [{"v": 1, "name": "x", "type": "u8"}])
    db.create_shape(2, 1, "large", [{"v": 1, "name": "x", "type": "u8"}, {"v": 2, "name": "y", "type": "u8"}])
    db.set_realm_default_shape(1, 2)
    db.create_classic_layout(1, 1, "c", shape_id=2)
    db.put(1, 1, [1, 2])                               # uses the default shape 2
    print(db.realms[1]["default_shape_id"], db.get_internal(1, 1)["by_v"])
    db.sync()                                          # written to the file, not committed
    print(db.last_valid_end > db.committed_end, db.next_block_id)

2 {1: 1, 2: 2}
True 4


*Erwartete Ausgabe (aus dem Cookbook):*

```text
2 {1: 1, 2: 2}
True 4
```

## 7. Einzelne Zellen, Lanes, Null und Löschen

Jede Zelle `(R, u, v, lane)` lässt sich direkt schreiben, auch außerhalb eines Shapes.
Wird die Zelle weder durch ein Shape-Feld noch durch eine Erweiterung beschrieben, gib ihren
Typ mit `type=` an (der Typ wird dann zusammen mit dem Wert gespeichert).

In [14]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "cells.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "grid")
    db.put_cell(1, 14, 744, "left", type="utf8")                # lane NL (default)
    db.put_cell(1, 14, 744, "right", type="utf8", lane="NR")    # same (u, v), other lane
    db.put_cell(1, 14, 745, None, type="utf8")                  # explicit null
    db.commit()

    print(db.get_value(1, 14, 744), db.get_value(1, 14, 744, lane="NR"))
    print(db.get_value(1, 14, 746))                              # never written -> None
    cell = db.get_cell(1, 14, 745, dyn_address=False)
    print(cell["present"], cell["value"])                        # null: present, value None
    print(db.get_cell(1, 14, 746, dyn_address=False))            # missing -> None

    db.delete_cell(1, 14, 744, commit=True)                      # tombstone in lane NL only
    print(db.get_value(1, 14, 744), db.get_value(1, 14, 744, lane="NR"))

left right
None
True None
None
None right


*Erwartete Ausgabe (aus dem Cookbook):*

```text
left right
None
True None
None
None right
```

Eine mit `None` geschriebene Zelle existiert: `present` ist `True`, und sie zählt in SQL als
Zelle des Datensatzes. Eine Zelle, die nie geschrieben oder gelöscht wurde, existiert nicht.

**Die drei Löschoperationen.** Alle arbeiten auf logischen Zellen: Ob eine Zelle
ein BASE-Feld oder eine Erweiterung ist, ob sie im dichten Datensatz oder als
einzelne Zelle gespeichert ist und ob sie einen Namen hat, macht keinen Unterschied.

* `delete_cell(realm_id, u, v)` löscht eine Zelle und gibt `True` zurück, wenn sie
  lebte (ein gespeichertes `None` eingeschlossen); andernfalls gibt es `False` zurück und schreibt
  nichts.
* `delete_area(realm_id, u_from, u_to, v_from, v_to)` löscht jede lebende
  Zelle im Rechteck (Grenzen eingeschlossen) und gibt ihre Zahl zurück. Es ist
  eine Einheit: Scheitert es (Größengrenze, E/A-Fehler), wird nichts davon geschrieben.
  Seine Kosten wachsen mit der Zahl der lebenden Zellen im Bereich.
* `delete_record(layout_id, object_id)` löscht jede lebende Zelle eines Datensatzes
  in einem beliebigen Layout und gibt ihre Zahl zurück.

Löschst du ein einzelnes Feld eines dichten Datensatzes mit `delete_cell`, liest es sich
als `None` (`present` ist `False`); die anderen Felder bleiben. Ein späteres `put` oder
`put_cell` schreibt die Zelle wieder.
Koordinaten müssen ganze Zahlen sein: `int`, NumPy-Ganzzahlen und ganzzahlige Floats wie
`3.0` werden akzeptiert, `2.7` löst `TypeError` aus (kein stilles Abschneiden).

Lanes können als `"NL"`/`"NR"` oder `0`/`1` angegeben werden (auch kleingeschrieben). Ohne
`type=` und ohne deklariertes Feld wird der Schreibvorgang abgelehnt:

In [15]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "t.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    try:
        db.put_cell(1, 1, 1, 42)
    except nhydb.SchemaError as e:
        print("SchemaError:", e)

SchemaError: R1,u1,v1: cannot derive a data type; specify type=...


*Erwartete Ausgabe (aus dem Cookbook):*

```text
SchemaError: R1,u1,v1: cannot derive a data type; specify type=...
```

**Viele Zellen auf einmal.** `put_points(realm_id, cells)` schreibt eine Liste von
Tupeln `(u, v, value)` oder `(u, v, value, type)`; `delete_area` löscht einen
Block von Zellen in einem Aufruf.

In [16]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "points.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "sensors")
    report = db.put_points(1, [
        (1, 1, 20.5, "f64"),
        (1, 2, 21.0, "f64"),
        (2, 1, "offline", "utf8"),
        (10**30, 7, b"\x00\x01", "bytes"),       # coordinates may be arbitrarily large
    ], commit=True)
    print(report["cells"], report["u_groups"])
    print(db.get_value(1, 1, 2), db.get_value(1, 10**30, 7))
    print(db.delete_area(1, 1, 2, 1, 2, commit=True))   # u1..u2, v1..v2: three cells
    print(db.get_value(1, 2, 1))

4 3
21.0 b'\x00\x01'
3
None


*Erwartete Ausgabe (aus dem Cookbook):*

```text
4 3
21.0 b'\x00\x01'
3
None
```

**Schreib- und Löschvorgänge sammeln: `batch()`.** Viele einzelne `put_cell`-Aufrufe
schreiben jeweils einen kleinen Block. `with db.batch(commit=False, u_window=10_000) as b:`
sammelt `b.put_cell(...)`, `b.delete_cell(...)`, `b.delete_record(...)` und
`b.delete_area(...)` (dieselben Argumente wie auf der Datenbank, ohne `commit`)
und schreibt sie am Ende des Blocks als eine Einheit – weniger, größere Blöcke und eine
kleinere Datei. Jeder Aufruf wird sofort geprüft (dieselben Fehler wie beim einzelnen
Aufruf); der Endzustand ist der, den die einzelnen Aufrufe in derselben Reihenfolge
hinterlassen würden (eine vor einem `delete_area` geschriebene Zelle wird von ihm gelöscht, eine danach
geschriebene bleibt); Lesezugriffe innerhalb des Blocks sehen die gesammelten
Operationen **nicht**; endet der Block mit einer Ausnahme, wird nichts geschrieben.
`commit=True` committet nach dem Schreiben. `b.flush()` schreibt vorzeitig und gibt
Zähler zurück (`"deleted"`: von allen Löschoperationen des Batches gelöschte lebende Zellen),
`b.discard()` verwirft die gesammelten Operationen, `len(b)` zählt sie.

In [17]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "batch.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "sensors")
    db.put_cell(1, 5, 1, 99.0, type="f64", commit=True)

    with db.batch(commit=True) as b:               # collect, write once at the end
        for u in range(1, 1001):
            b.put_cell(1, u, 1, u / 10, type="f64")
        b.put_cell(1, 7, 1, -1.0, type="f64")      # last call per cell wins
        b.delete_cell(1, 5, 1)                     # written as a tombstone at the end
        print(len(b), db.get_value(1, 7, 1))       # collected calls; no read-your-writes inside the block
    print(db.get_value(1, 7, 1), db.get_value(1, 5, 1), db.get_value(1, 1000, 1))

    try:
        with db.batch() as b:
            b.put_cell(1, 2000, 1, 1.0, type="f64")
            b.put_cell(1, 2001, 1, "text", type="f64")   # checked at the call
    except (TypeError, ValueError) as e:
        print(type(e).__name__)
    print(db.get_value(1, 2000, 1))                # an exception writes nothing

1002 None
-1.0 None 100.0
ValueError
None


*Erwartete Ausgabe (aus dem Cookbook):*

```text
1002 None
-1.0 None 100.0
ValueError
None
```

> **Hinweis:** `get(realm, u)` und `get_record(...)` liefern **dichte**
> Datensätze. Ein `u`, das nur dünn besetzte Zellen hat (geschrieben mit `put_cell`/
> `put_points`), liefert dort `None`; lies solche Zellen mit `get_value` oder
> `get_cell`, oder durchlaufe sie mit `iter_materialized_cells` (Abschnitt 12).

## 8. Erweiterungen: zusätzliche Felder pro Datensatz

Erweiterungen sind benannte, typisierte Felder eines Realms bei einem `v` oberhalb der BASE-
Felder. Sie werden dünn besetzt gespeichert, also bezahlen nur Datensätze mit einem Wert
dafür. Mit `base_reserve_to` reservierst du einen `v`-Bereich für künftige BASE-Felder.

In [18]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "ext.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "products", base_reserve_to=20)
    db.create_shape(1, 1, "product", [
        {"v": 1, "name": "sku", "type": "utf8"},
        {"v": 2, "name": "price", "type": "f64"},
    ])
    db.create_classic_layout(1, 1, "products", shape_id=1)
    db.define_extension(1, 30, "utf8", name="color")
    db.define_extensions(1, [{"v": 31, "name": "weight_kg", "type": "f64"},
                             {"v": 32, "name": "tags", "type": "utf8"}])
    db.put_dense_batch(1, 1, [["A-100", 9.99], ["B-200", 24.5], ["C-300", 3.2]])

    db.put_extensions(1, 1, {"color": "red", "weight_kg": 0.4})       # by name
    db.put_extension_list(1, 2, [30, 31], ["blue", 1.25])             # by v
    db.put_extension_range(1, 3, 30, ["green", 0.1, "sale"])          # v = 30, 31, 32
    db.commit()

    print(db.get(1, 1))
    print(db.get(1, 3, include_extensions=False))
    print([e["name"] for e in db.extension_catalog(1)], db.resolve_v(1, "tags"))

    db.put_extensions(1, 1, {"tags": "new"}, merge=False, commit=True)  # replace all extensions
    print({k: v for k, v in db.get(1, 1).items() if not k.startswith("_")})

{'_realm': 1, '_u': 1, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'sku': 'A-100', 'price': 9.99, 'color': 'red', 'weight_kg': 0.4}
{'_realm': 1, '_u': 3, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'sku': 'C-300', 'price': 3.2}
['color', 'weight_kg', 'tags'] 32
{'sku': 'A-100', 'price': 9.99, 'tags': 'new'}


*Erwartete Ausgabe (aus dem Cookbook):*

```text
{'_realm': 1, '_u': 1, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'sku': 'A-100', 'price': 9.99, 'color': 'red', 'weight_kg': 0.4}
{'_realm': 1, '_u': 3, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'sku': 'C-300', 'price': 3.2}
['color', 'weight_kg', 'tags'] 32
{'sku': 'A-100', 'price': 9.99, 'tags': 'new'}
```

Mit `merge=True` (Standard) werden nur die angegebenen Erweiterungen geschrieben; mit
`merge=False` werden die übrigen aktiven Erweiterungen des Datensatzes gelöscht.
`put_record` und `put_dense_batch` (über `extensions_by_u`) können
Erweiterungen zusammen mit den BASE-Feldern schreiben:

In [19]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "ext2.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "name", "type": "utf8"}])
    db.create_classic_layout(1, 1, "c", shape_id=1)
    db.define_extension(1, 10, "utf8", name="note")
    db.put_record(1, 1, {"name": "x", "note": "via put_record"})
    db.put(1, 2, ["y"], extensions={"note": "via put"})
    db.put_dense_batch(1, 3, [["z"]], extensions_by_u={3: {"note": "via batch"}})
    db.commit()
    print([db.get_value(1, u, 10) for u in (1, 2, 3)])

['via put_record', 'via put', 'via batch']


*Erwartete Ausgabe (aus dem Cookbook):*

```text
['via put_record', 'via put', 'via batch']
```